# 06 - Model zona GABUNGAN, 4 kelas

Notebook 05 membuat model **baru** dari satu dataset, dan hasilnya: jawaban salah
hilang (2 -> 0) tetapi **cakupan zona runtuh 7/16 -> 2/16** dan latensi dua kali lipat
(10,2 -> 20,5 detik). Sebabnya ia tidak mewarisi apa pun dari `best_zone.pt`.

Notebook ini **menggabungkan** keduanya: dataset 4 kelas yang memberi kemampuan menolak
kode produksi, PLUS dataset 1 kelas yang dipakai `best_zone.pt` - kelas tunggalnya
dipetakan ke `date`. Tujuannya satu model yang punya cakupan lama DAN kelas baru.

Mengikuti notebook 03: augmentasi rotasi (notebook 05 **tidak punya** augmentasi sama
sekali, dan itu salah satu tersangka cakupan yang runtuh), dedupe pHash antar-dataset,
dan semua hasil ditulis ke Drive.


In [ ]:
# Sel 1 - GPU dan Drive
import torch
assert torch.cuda.is_available(), (
    'Runtime TANPA GPU. Runtime > Change runtime type > T4 GPU, lalu jalankan ulang.')
print('GPU:', torch.cuda.get_device_name(0))
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Sel 2 - pustaka
%pip -q install ultralytics roboflow imagehash
import ultralytics
print('ultralytics', ultralytics.__version__)


In [ ]:
# Sel 3 - folder kerja di Drive (pola notebook 01/03/04)
from pathlib import Path
DRIVE_DIR  = Path('/content/drive/MyDrive/sayit')
RUNS_DIR   = DRIVE_DIR / 'runs'
MODELS_DIR = DRIVE_DIR / 'models'
HASIL_DIR  = DRIVE_DIR / 'hasil'
for d in (RUNS_DIR, MODELS_DIR, HASIL_DIR):
    d.mkdir(parents=True, exist_ok=True)
print('Folder kerja:', DRIVE_DIR)


In [ ]:
# Sel 4 - konfigurasi.
#
# Kunci dari Colab Secrets bernama ROBOFLOW_API_KEY (ikon kunci di bilah kiri),
# saklar 'Notebook access' MENYALA. WAJIB kunci PRIVATE - publishable key
# berawalan rf_ ditolak 401.
PESAN = ('Secret ROBOFLOW_API_KEY tidak terbaca. Buka ikon kunci di bilah kiri, '
         'buat secret bernama persis ROBOFLOW_API_KEY, isi dengan Private API Key, '
         'lalu nyalakan saklar Notebook access.')
try:
    from google.colab import userdata
    RF_API_KEY = userdata.get('ROBOFLOW_API_KEY')
except Exception as e:
    raise SystemExit(f'{PESAN} Pesan aslinya: {e}')
assert RF_API_KEY, PESAN

# Tiap entri: workspace/project/versi, plus PEMETAAN nama kelas dataset itu ke
# kosakata kita. Dataset 1 kelas dipetakan ke 'date'. Kelas yang tidak ada di
# peta akan DIBUANG dan jumlahnya dilaporkan - lebih baik hilang nyaring
# daripada diam-diam jadi kelas yang salah (jebakan label tertukar, 27-09).
KELAS = ['date', 'code', 'due', 'prod']      # urutan ini jadi indeks YOLO

DATASETS = [
    dict(nama='ESIGELEC', workspace='esigelec-rniez', project='expiry', versi=1,
         peta={'date': 'date', 'code': 'code', 'due': 'due', 'prod': 'prod'}),
    # Dataset yang dipakai best_zone.pt. LISENSINYA TIDAK TERCANTUM di Roboflow -
    # periksa sebelum masuk jurnal, atau ganti dengan salah satu di bawah.
    dict(nama='COOL', workspace='coolstuff-rfnc5', project='expiry-date-detection-i3bav',
         versi=1, peta={'Label': 'date'}),
    # Alternatif berlisensi jelas, aktifkan bila COOL bermasalah:
    # dict(nama='ELD', workspace='expiry-label-detection',
    #      project='expiry-date-detection-egbf3', versi=2,   # CC BY 4.0, 1.014 gambar
    #      peta={'Label': 'date'}),
    # dict(nama='ABHI', workspace='abhilashasacademicworkspace-8igt2',
    #      project='expiry-date-detection-ssxnm', versi=2,   # MIT, 5.229 gambar
    #      peta={'date': 'date'}),
]

EPOCHS   = 100
PATIENCE = 20
IMGSZ    = 640
BATCH    = 16
SEED     = 42
PHASH_MAX_DIST = 8
RUN_NAME = 'zone_gabungan_v1'

# Augmentasi, disalin dari notebook 03. Pelajaran model uang: bawaan ultralytics
# degrees=0.0 membuat model tidak pernah melihat objek miring - dan kemasan di
# tangan pengguna tunanetra hampir tidak pernah tegak. Notebook 05 melewatkan ini.
AUG = dict(degrees=15.0, translate=0.1, scale=0.5, fliplr=0.0, mosaic=1.0)
print('kunci terbaca,', len(RF_API_KEY), 'karakter')


In [ ]:
# Sel 5 - unduh SEMUA dataset lewat API (tidak ada unduhan manual)
from roboflow import Roboflow
import yaml
rf = Roboflow(api_key=RF_API_KEY)
for d in DATASETS:
    ds = rf.workspace(d['workspace']).project(d['project']).version(d['versi']).download('yolov8')
    d['lokasi'] = Path(ds.location)
    d['nama_kelas'] = yaml.safe_load((d['lokasi'] / 'data.yaml').read_text())['names']
    print(f"{d['nama']:10} {d['lokasi']}  kelas={d['nama_kelas']}")


In [ ]:
# Sel 6 - PERIKSA PEMETAAN sebelum menggabung.
# Kelas yang tidak ada di peta akan dibuang; kalau jumlahnya besar, hentikan.
for d in DATASETS:
    tak_dipetakan = [n for n in d['nama_kelas'] if n not in d['peta']]
    print(f"{d['nama']:10} dipetakan: {d['peta']}")
    if tak_dipetakan:
        print(f"{'':10} DIBUANG  : {tak_dipetakan}")
    salah = [v for v in d['peta'].values() if v not in KELAS]
    assert not salah, f"{d['nama']}: peta menunjuk kelas yang tidak ada di KELAS: {salah}"


In [ ]:
# Sel 7 - baca semua record, petakan kelasnya, dedupe pHash antar-dataset.
# Dedupe penting karena dataset Roboflow sering saling menyalin: tanpa ini satu
# foto bisa muncul di train SEKALIGUS di val, dan angkanya jadi optimistis.
import collections
import imagehash
from PIL import Image

records, dibuang = [], collections.Counter()
for d in DATASETS:
    idx2nama = {i: n for i, n in enumerate(d['nama_kelas'])}
    for split in ('train', 'valid', 'test'):
        img_dir = d['lokasi'] / split / 'images'
        if not img_dir.is_dir():
            continue
        for g in sorted(img_dir.iterdir()):
            if g.suffix.lower() not in ('.jpg', '.jpeg', '.png'):
                continue
            t = d['lokasi'] / split / 'labels' / (g.stem + '.txt')
            kotak = []
            for b in (t.read_text().splitlines() if t.exists() else []):
                if not b.strip():
                    continue
                bagian = b.split()
                asal = idx2nama.get(int(bagian[0]))
                tujuan = d['peta'].get(asal)
                if tujuan is None:
                    dibuang[f"{d['nama']}/{asal}"] += 1
                    continue
                kotak.append((KELAS.index(tujuan), [float(v) for v in bagian[1:5]]))
            if kotak:          # gambar tanpa kotak TIDAK diikutkan: dalam YOLO ia
                records.append(dict(dataset=d['nama'], path=g, boxes=kotak))

print(f'{len(records)} gambar berkotak')
print('kotak dibuang karena kelas tak dipetakan:', dict(dibuang) or 'tidak ada')

# pHash: gambar yang jaraknya <= PHASH_MAX_DIST dianggap satu kelompok
for r in records:
    r['hash'] = imagehash.phash(Image.open(r['path']).convert('RGB'))
kelompok, sisa = [], list(records)
while sisa:
    inti = sisa.pop(0)
    grup, lain = [inti], []
    for r in sisa:
        (grup if (inti['hash'] - r['hash']) <= PHASH_MAX_DIST else lain).append(r)
    kelompok.append(grup)
    sisa = lain
print(f'{len(kelompok)} kelompok kemiripan dari {len(records)} gambar',
      f'-> {len(records) - len(kelompok)} kemungkinan duplikat')


In [ ]:
# Sel 8 - bagi per KELOMPOK (bukan per berkas) lalu tulis dataset YOLO
import random, shutil
rng = random.Random(SEED)
rng.shuffle(kelompok)
n = len(kelompok)
batas_train, batas_val = int(0.7 * n), int(0.9 * n)
bagian = {'train': kelompok[:batas_train],
          'val':   kelompok[batas_train:batas_val],
          'test':  kelompok[batas_val:]}

ROOT = Path('/content/data_zona_gabungan')
if ROOT.exists():
    shutil.rmtree(ROOT)
hitung = collections.Counter()
for split, grup_split in bagian.items():
    (ROOT / split / 'images').mkdir(parents=True)
    (ROOT / split / 'labels').mkdir(parents=True)
    i = 0
    for grup in grup_split:
        for r in grup:
            stem = f"{r['dataset']}_{i:05d}"
            i += 1
            shutil.copy(r['path'], ROOT / split / 'images' / (stem + r['path'].suffix))
            isi = []
            for k, b in r['boxes']:
                isi.append(str(k) + ' ' + ' '.join(f'{v:.6f}' for v in b))
                hitung[(split, KELAS[k])] += 1
            (ROOT / split / 'labels' / (stem + '.txt')).write_text('\n'.join(isi))
    print(f'{split:6} {i:5} gambar dari {len(grup_split)} kelompok')

YAML = ROOT / 'data.yaml'
YAML.write_text(yaml.safe_dump({'path': str(ROOT), 'train': 'train/images',
                                'val': 'val/images', 'test': 'test/images',
                                'nc': len(KELAS), 'names': KELAS}))
print()
for (split, k), v in sorted(hitung.items()):
    print(f'  {split:6} {k:6} {v:5} kotak')


In [ ]:
# Sel 9 - latih, langsung ke Drive
from ultralytics import YOLO
RUN_DIR = RUNS_DIR / RUN_NAME
BEST_PT = RUN_DIR / 'weights' / 'best.pt'
model = YOLO('yolov8n.pt')
model.train(data=str(YAML), epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, patience=PATIENCE,
            seed=SEED, deterministic=True, project=str(RUNS_DIR), name=RUN_NAME,
            exist_ok=True, plots=True, **AUG)
print('save_dir :', model.trainer.save_dir)
assert BEST_PT.exists(), f'Latihan selesai tetapi {BEST_PT} tidak ada.'


In [ ]:
# Sel 10 - evaluasi dan tulis metrik ke Drive
import json
from datetime import datetime
terbaik = YOLO(str(BEST_PT))
m = terbaik.val(data=str(YAML), split='test', imgsz=IMGSZ,
                project=str(RUNS_DIR), name=RUN_NAME + '_test', exist_ok=True)
per_kelas = {}
for i, k in enumerate(KELAS):
    try:
        per_kelas[k] = round(float(m.box.ap50[i]), 4)
    except (IndexError, TypeError):
        pass
ringkas = {'run': RUN_NAME, 'waktu': datetime.now().isoformat(timespec='seconds'),
           'datasets': [f"{d['workspace']}/{d['project']} v{d['versi']}" for d in DATASETS],
           'gambar': len(records), 'kelompok': len(kelompok),
           'epochs': EPOCHS, 'imgsz': IMGSZ, 'aug': AUG,
           'mAP50': round(float(m.box.map50), 4),
           'mAP50_95': round(float(m.box.map), 4),
           'ap50_per_kelas': per_kelas, 'bobot': str(BEST_PT)}
(HASIL_DIR / f'{RUN_NAME}_metrik.json').write_text(json.dumps(ringkas, indent=2),
                                                   encoding='utf-8')
print(json.dumps(ringkas, indent=2))


In [ ]:
# Sel 11 - salin bobot ke Drive dengan nama bertanggal
mb = BEST_PT.stat().st_size / 1e6
print(f'{BEST_PT} = {mb:.1f} MB ->',
      'selesai normal' if mb < 15 else 'TERPUTUS, jangan dipakai')
stamp = datetime.now().strftime('%Y%m%d_%H%M')
dst = MODELS_DIR / f'best_zone_gabungan_{stamp}.pt'
shutil.copy(BEST_PT, dst)
print('Bobot disalin ke', dst)


## Sesudah ini

Salin bobotnya ke `models/` di laptop (**jangan timpa** `best_zone.pt`), lalu ukur:

```
python scripts\ukur_kedaluwarsa.py
```

Tiga angka yang dibandingkan, terhadap dasar yang sudah ada:

| bobot | BENAR/SALAH/DIAM | median | zona ketemu |
|---|---|---|---|
| `best_zone.pt` (1 kelas) | 13 / **2** / 1 | **10,24 s** | **7/16** |
| 4 kelas, satu dataset | 13 / **0** / 3 | 20,46 s | 2/16 |
| gabungan | ? | ? | ? |

**Yang dicari: 0 SALAH seperti kolom tengah, DAN cakupan zona + latensi seperti kolom
kiri.** Kalau cakupan tetap rendah, berarti penyebabnya bukan dataset melainkan
`zone_score_min` 0,40 yang dikalibrasi untuk model 1 kelas - itu sapuan berikutnya,
dan jauh lebih murah daripada melatih ulang.

**Lisensi:** `esigelec-rniez/expiry` CC BY 4.0. `coolstuff-rfnc5/...` **tidak
mencantumkan lisensi** - periksa sebelum masuk jurnal, atau ganti dengan `ELD`
(CC BY 4.0) / `ABHI` (MIT) yang sudah disiapkan terkomentar di Sel 4.
